Genetic Algorithm

Helper Functions

In [ ]:
import numpy as np
import pandas as pd

def top_k(arr, k):
    return np.argpartition(arr, -k)[-k:]

def elitism(e, population, fitnesses):
    top = top_k(fitnesses, e)
    return population[top]

class Selector(object):
    def __init__(self):
        super().__init__()

    def prep_selector(self, fitnesses):
        pass

    def select(self, population):
        pass


class RouletteWheelSelector(Selector):
    def __init__(self):
        super().__init__()
        self.table = []

    def prep_selector(self, fitnesses):
        transformed = np.exp(fitnesses)
        total = np.sum(transformed)
        self.table = transformed / total
        self.indicies = np.arange(0, len(fitnesses), 1, dtype=int)

    def select(self, population):
        parent1_idx, parent2_idx = np.random.choice(self.indicies, size=2, replace=False, p=self.table)
        parent1 = population[parent1_idx, :]
        parent2 = population[parent2_idx, :]
        return parent1, parent2

class TournamentSelector(Selector):
    def __init__(self):
        super().__init__()
        self.table = []
    
    def prep_selector(self, fitnesses):
        self.table = fitnesses
    
    def select(self,population):
        indicies = np.arange(0, len(population), 1, dtype=int)
        sub_population = population[indicies]
        sub_pop_fitnesses = self.table[indicies]
        parent1_idx, parent2_idx = top_k(sub_pop_fitnesses, 2)
        return sub_population[parent1_idx], sub_population[parent2_idx]

def single_point(parent1, parent2):
    point = np.random.randint(1, len(parent1))
    child = np.zeros_like(parent1)
    child[:point] = parent1[:point]
    child[point:] = parent2[point:]
    return child


def interpolation(parent1, parent2):
    lambda_point = np.random.uniform()
    return lambda_point * parent1 +  (1 - lambda_point) * parent2

def uniform(parent1, parent2):
    selection = np.random.randint(0, 2, size=len(parent1))
    neg_selection = 1 - selection
    child = parent1 * selection + parent2 * neg_selection
    return child

#Ensures that crossing over dna returns an array with unique elements
def order_crossover(parent1, parent2):
    n = len(parent1)
    start = np.random.randint(0, n - 1)
    end = np.random.randint(start + 1, n)
    child = np.full(n, -1, dtype=int)
    child[start:end] = parent1[start:end]
    fill = [v for v in parent2 if v not in child[start:end]]
    idxs = [i for i in range(n) if child[i] == -1]
    for i, v in zip(idxs, fill):
        child[i] = v
    return child

class Game:
    def __init__(self, name, platform, genre, year, na, eu, jp, ww):
        self.name = name
        self.platform = platform
        self.genre = genre
        self.release = year
        self.nasales = na
        self.eusales = eu
        self.jpsales = jp
        self.wwsales = ww

class Company:
    def __init__(self, name):
        self.name = name
        self.sales2024 = 0
        self.sales2026 = 0
        self.games_2024 = []
        self.games_2026 = []

    def add_game(self, Game, year):
        total_sales = (
                Game.nasales +
                Game.eusales +
                Game.jpsales +
                Game.wwsales
            )
        
        if year == 2024:
            self.games_2024.append(Game)
            self.sales2024 += total_sales
        elif year == 2026:
            self.games_2026.append(Game)
            self.sales2026 += total_sales
    
    @property
    def profit(self):
        return self.sales2026 - self.sales2024
    
    @property
    def title_count(self):
        return len(self.games_2024) + len(self.games_2026)
    
    @property
    def genre_diversity(self):
        all_games = self.games_2024 + self.games_2026
        if not all_games:
            return 0.0
        genres = {g.genre for g in all_games}
        return len(genres) / len(all_games)
    
    @property
    def platform_count(self):
        all_games = self.games_2024 + self.games_2026
        return len({g.platform for g in all_games})
    
    @property
    def avg_sales_per_title(self):
        return self.total_sales / self.title_count if self.title_count > 0 else 0.0
    
    @property
    def total_sales(self):
        return self.sales2026
    
    def feature_array(self):
        total = self.total_sales if self.total_sales > 0 else 1e-9
        na = sum(g.nasales for g in self.games_2024 + self.games_2026)
        eu = sum(g.eusales for g in self.games_2024 + self.games_2026)
        jp = sum(g.jpsales for g in self.games_2024 + self.games_2026)
        return np.array([
            self.total_sales,          # 0 total_sales
            na / total,                # 1 na_ratio
            eu / total,                # 2 eu_ratio
            jp / total,                # 3 jp_ratio
            self.avg_sales_per_title,  # 4 avg_sales_title
            self.title_count,          # 5 title_count
            self.genre_diversity,      # 6 genre_diversity
            float(self.platform_count),# 7 platform_count
        ], dtype=float)
    
    def __repr__(self):
        return (
            f"Company(name={self.name}, "
            f"sales2024={self.sales2024:.2f}, "
            f"sales2026={self.sales2026:.2f}, "
            f"profit={self.profit:.2f})"
        )

def load_data(df, companies, year):
    for _, row in df.iterrows():
        publisher = row['Publisher']
    
        if publisher not in companies:
            companies[publisher] = Company(name=publisher)
        
        game = Game(
            row['Name'],
            row['Platform'],
            row['Genre'],
            row['Release_Year'],
            row['NA_Sales'],
            row['EU_Sales'],
            row['JP_Sales'],
            row['Other_Sales']
        )
    
        companies[publisher].add_game(game, year)

Random Data Sampling for testing

In [ ]:
df_2024 = pd.read_csv("../data/vgData2024.csv")
df_2026 = pd.read_csv("../data/vgData2026.csv")

df_2024_RS=df_2024.sample(n=100, random_state=250)
df_2026_RS=df_2026.sample(n=100, random_state=250)
#With both datasets having the same entires,
#the same seed would return the exact same entires in both datasets

companies = {}

load_data(df_2024_RS, companies, year=2024)
load_data(df_2026_RS, companies, year=2026)

for company in companies.values():
    if(company.profit > 0):
        print(company)

Agent Code

In [ ]:
CDNA_FEATURES = [
    "total_sales",
    "na_market_ratio",
    "eu_market_ratio",
    "jp_market_ratio",
    "avg_sales_per_title",
    "title_count",
    "genre_diversity",
    "platform_count",
]
N_CDNA = len(CDNA_FEATURES)

class Agent:

    def __init__(self, cdna=None, dna=None, mutate_cdna=False,
        n_companies=10, companies=None, company_index=None):
        self.mutate_cdna = mutate_cdna
        self.n_companies = n_companies
        self._fitness = None
        
        self.companies = companies      # dict: name → Company
        self.company_index = company_index  # list: int → name
        
        if cdna is not None:
            self.cdna = np.array(cdna, dtype=int)
        else:
            self.cdna = np.zeros(N_CDNA, dtype=int)
            n_active = np.random.randint(1, N_CDNA + 1)
            chosen = np.random.choice(N_CDNA, n_active, replace=False)
            self.cdna[chosen] = 1
        
        if dna is not None:
            self.dna = np.array(dna, dtype=int)
        else:
            self.dna = np.random.permutation(n_companies)


    def score_company(self, company, feature_weights: np.ndarray | None = None):
        fv = company.feature_array()           # raw feature values
        mask = self.cdna.astype(float)            # 0/1 gate per feature
        scaled = _normalise_features(fv)            # bring all features to [0,1]

        if feature_weights is not None:
            return float(np.dot(scaled * mask, feature_weights))
        return float(np.sum(scaled * mask))


    @property
    def fitness(self):
        return self._fitness
    
    @fitness.setter
    def fitness(self, value):
        self._fitness = float(value)
    
    @property
    def active_features(self):
        return [CDNA_FEATURES[i] for i, g in enumerate(self.cdna) if g == 1]
    
    @property
    def top_company(self):
        return int(self.dna[0])
    
    def classify(self, company):
        n_active = int(self.cdna.sum())
        if n_active == 0:
            return 0
        score = self.score_company(company)
        return int((score / n_active) >= 0.5)
    
    @property
    def predictions(self):
        return {
            name: self.classify(self.companies[name])
            for name in self.company_index
        }
    
    def mutate(self, cdna_rate=0.05, dna_rate=0.1):
        new_cdna = self.cdna.copy()
        new_dna = self.dna.copy()
        
        if self.mutate_cdna:
            flips = np.random.random(N_CDNA) < cdna_rate
            new_cdna = np.where(flips, 1 - new_cdna, new_cdna)
            if new_cdna.sum() == 0:
                new_cdna[np.random.randint(N_CDNA)] = 1
        
        if np.random.random() < dna_rate:
            i, j = np.random.choice(self.n_companies, 2, replace=False)
            new_dna[i], new_dna[j] = new_dna[j], new_dna[i]
        
        return Agent(new_cdna, new_dna, self.mutate_cdna, self.n_companies,
            self.companies, self.company_index)
    
    def crossover(self, other):
        mask = np.random.randint(0, 2, N_CDNA).astype(bool)
        c1 = np.where(mask,  self.cdna, other.cdna)
        c2 = np.where(mask, other.cdna,  self.cdna)
        for c in (c1, c2):
            if c.sum() == 0:
                c[np.random.randint(N_CDNA)] = 1
        
        d1 = order_crossover(self.dna,  other.dna)
        d2 = order_crossover(other.dna, self.dna)
        
        return (Agent(c1, d1, self.mutate_cdna, self.n_companies,
            self.companies, self.company_index),
            Agent(c2, d2, self.mutate_cdna, self.n_companies,
            self.companies, self.company_index))

    def __repr__(self):
        fit = f"{self._fitness:.4f}" if self._fitness is not None else "unevaluated"
        return (
            f"Agent(cdna={self.cdna.tolist()}, "
            f"active={self.active_features}, "
            f"top_co={self.top_company}, fitness={fit})"
        )

_feature_stats: dict = {}

def build_feature_stats(companies):
    vectors = np.array([c.feature_array() for c in companies.values()])
    _feature_stats['min'] = vectors.min(axis=0)
    _feature_stats['max'] = vectors.max(axis=0)

def _normalise_features(fv):
    if not _feature_stats:
        return fv
    mn, mx = _feature_stats['min'], _feature_stats['max']
    rng = np.where(mx - mn == 0, 1.0, mx - mn)
    return (fv - mn) / rng

def evaluate_fitness(agent, companies, company_index):
    n = len(company_index)
    correct = 0

    for i in range(n):
        company = companies[company_index[i]]
        predicted = agent.classify(company)
        actual = int(company.profit > 0)
        if predicted == actual:
            correct += 1
    
    accuracy = correct / n
    
    return accuracy

#Function to separate testing cell from Agent cell
def run_ga(
        companies,
        company_index,
        selector,
        pop_size,
        n_generations,
        elite_k,
        cdna_rate,
        dna_rate,
        mutate_cdna
    ):
    n_companies = len(company_index)
    build_feature_stats(companies)
    
    population = [
        Agent(mutate_cdna=mutate_cdna, n_companies=n_companies,
        companies=companies, company_index=company_index)
        for _ in range(pop_size)
    ]
    
    fitness_history = []
    best_agent = None

    print("\nTraining Generations\n")
    
    for gen in range(n_generations):
    
        fitnesses = np.array([
            evaluate_fitness(agent, companies, company_index)
            for agent in population
        ])
        for agent, f in zip(population, fitnesses):
            agent.fitness = f
        
        best_idx = int(np.argmax(fitnesses))
        best_agent = population[best_idx]
        fitness_history.append(fitnesses[best_idx])
        
        print(
            f"Gen {gen} \n best fitness: {fitnesses[best_idx]:.4f} "
            f"\n mean: {fitnesses.mean():.4f} "
            f"\n active feats: {best_agent.active_features}"
        )
        
        elite_indices = top_k(fitnesses, elite_k)
        elites = [population[i] for i in elite_indices]
        
        pop_array = np.array([a.dna for a in population])
        selector.prep_selector(fitnesses)
        
        children = []
        while len(children) < pop_size - elite_k:
            p1_dna, p2_dna = selector.select(pop_array)
            p1 = next(a for a in population if np.array_equal(a.dna, p1_dna))
            p2 = next(a for a in population if np.array_equal(a.dna, p2_dna))
            c1, c2 = p1.crossover(p2)
            children.extend([c1.mutate(cdna_rate, dna_rate),
                c2.mutate(cdna_rate, dna_rate)])
        
        population = elites + children[:pop_size - elite_k]
    
    return best_agent, fitness_history

Testing Agents

In [ ]:
print(f"Companies loaded: {len(companies)}")

profitable_companies = 0;
for c in companies.values():
    if c.profit > 0:
        profitable_companies = profitable_companies + 1
        
print(f"Profitable companies: {profitable_companies}")

for c in companies.values():
    if c.profit > 0:
        print(f"  {c}")

company_index = list(companies.keys())

best, history = run_ga(
    companies = companies,
    company_index = company_index,
    selector = TournamentSelector(),
    pop_size = 100,
    n_generations = 100,
    elite_k = 5,
    cdna_rate = 0.005,
    dna_rate = 0.005,
    mutate_cdna = True,
)

print(f"\nBest agent")
print(best)

top_name = company_index[best.top_company]
top_co = companies[top_name]
print(f"\nPredicted most profitable company : {top_name}")
print(f"  sales2024            = {top_co.sales2024:.2f}")
print(f"  sales2026            = {top_co.sales2026:.2f}")
print(f"  profit               = {top_co.profit:.4f}")
print(f"  is actually profitable: {top_co.profit > 0.0}")

print(f"\nTop 10 profitable companies:")
ranked = sorted(companies.values(), key=lambda c: c.profit, reverse=True)
for i, c in enumerate(ranked[:10], 1):
    marker = " <- predicted #1" if c.name == top_name else ""
    print(f"  {i}. {c}{marker}")

print(f"\nClassifications:")
preds = best.predictions
correct = sum(1 for name, pred in preds.items()
              if pred == int(companies[name].profit > 0))
print(f"  Accuracy: {correct}/{len(preds)} = {correct/len(preds):.1%}")
print(f"  {'Company':<40} {'Predicted':>10} {'Actual':>10} {'Match':>6}")
print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*6}")
for name, pred in preds.items():
    actual = int(companies[name].profit > 0)
    flag   = "Match" if pred == actual else "No Match"
    print(f"  {name:<40} {pred:>10} {actual:>10} {flag:>6}")